### Import the libraries

In [114]:
import spacy
from spacy import tokenizer
from bs4 import BeautifulSoup
import nltk
import string
import re
from nltk.stem import WordNetLemmatizer
from nltk.tokenize.toktok import ToktokTokenizer
import CONTRACTION_MAP
import pandas as pd


### Load the covid19 tweet data

In [115]:
rawData =pd.read_excel("COVID_19_vaccine_100.xlsx")
rawData.columns=['tweets']
rawData.head(5)

,tweets
0,@johensley @DarcyShepherd13 @JeffreyGuterman @...
1,May i remind you that the vaccine isnt just fo...
2,"@MJAckermanMDPhD\nHi, Dr. Ackerman. I will lik..."
3,"Shandro, Hinshaw to give COVID-19 vaccine upda..."
4,You ever Noticed This? They just announced Yes...


## Text Processing

#### Removing html tags

<span style="color:white">Often, unstructured text contains a lot of noise, especially if you use techniques like web or screen scraping. HTML tags are typically one of these components which don’t add much value towards understanding and analyzing text.</span>

In [116]:
def strip_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    stripped_text = soup.get_text()
    return stripped_text
strip_html_tags('<html><h2>May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO</h2></html>')

'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO'

#### Removing accented characters
<span style="color:white">Usually in any text corpus, you might be dealing with accented characters/letters, especially if you only want to analyze the English language. Hence, we need to make sure that these characters are converted and standardized into ASCII characters. A simple example — converting é to e.</span>

In [117]:
def remove_accented_chars(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    return text

remove_accented_chars('Sómě Áccěntěd těxt')
remove_accented_chars("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO'

#### Expanding Contractions
<span style="color:white">Contractions are shortened version of words or syllables. They often exist in either written or spoken forms in the English language. These shortened versions or contractions of words are created by removing specific letters and sounds. In case of English contractions, they are often created by removing one of the vowels from the word. Examples would be, do not to don’t and I would to I’d. Converting each contraction to its expanded, original form helps with text standardization.</span>

In [118]:
def expand_contractions(text, contraction_mapping=CONTRACTION_MAP.CONTRACTION_MAP):
    modified_text=[]
    for words in text.split(' '):
        val = CONTRACTION_MAP.CONTRACTION_MAP.get(words)
        if val is not None:
            modified_text.append(val)
        else:
            modified_text.append(words)
    modified_text=' '.join(modified_text)
    return modified_text

expand_contractions("Y'all are enjoying this class we'd think. It is so cool isnt it?")

"Y'all are enjoying this class we would think. It is so cool is not it?"

#### Removing Special Characters
<span style="color:white">Special characters and symbols are usually non-alphanumeric characters or even occasionally numeric characters (depending on the problem), which add to the extra noise in unstructured text. Usually, simple regular expressions (regexes) can be used to remove them.</span>

In [119]:
def remove_special_characters(text, remove_digits=True):
    text = ''.join([word.lower() for word in text if word not in string.punctuation])
    text = ''.join([re.sub(r'\d+','',word) for word in text if remove_digits])
    return text

remove_special_characters("Well this was fun! What do you think? 123#@!\\//4", remove_digits=True)
#remove_special_characters("May i remind you that the vaccine isnt just for those who @caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'well this was fun what do you think '

#### Removing Stopwords
<span style="color:white">Words which have little or no significance, especially when constructing meaningful features from text, are known as stopwords or stop words. These are usually words that end up having the maximum frequency if you do a simple term or word frequency in a corpus. Typically, these can be articles, conjunctions, prepositions and so on. Some examples of stopwords are a, an, the, and the like.</span>

In [120]:

stopword_list = nltk.corpus.stopwords.words('english')
tokenizer = ToktokTokenizer()
def remove_stopwords(text, is_lower_case=False):
    tokens = tokenizer.tokenize(text)
    tokens = [token.strip() for token in tokens]
    if is_lower_case:
        filtered_tokens = [token for token in tokens if token not in stopword_list]
    else:
        filtered_tokens = [token for token in tokens if token.lower() not in stopword_list]
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text

remove_stopwords("Let us see if we can or can not remove against the stopwords from a sentence.")
remove_stopwords("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'May remind vaccine isnt caught covid-19 , also prevent ................ https://t.co/htFc4jP4CO'

#### Remomving URLS
<span style="color:white">Words which contains urls are mostly not required for the data analysis</span>

In [121]:
def url_removal(text):
    modified_text = ' '.join([words for words in text.split(' ') if words[0:4]!="http"])
    return modified_text

url_removal("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO" )


'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................'

### Lemmatization

In [122]:
wl =WordNetLemmatizer()
def lemmatize_text(text):
    text = ' '.join([wl.lemmatize(i) for i in text.split(' ')])
    return text


### Stemming

In [123]:
ps = nltk.PorterStemmer()
def stem_text(text):
    text = ' '.join([ps.stem(i) for i in text.split(' ')])
    return text

In [137]:
def normalize_corpus(doc, html_stripping=True, contraction_expansion=True,
                     accented_char_removal=True, text_lower_case=True, special_char_removal=True,
                     stopword_removal=True,remove_url=True, lemmatize=True,stem=True,remove_digits=True):

    normalized_corpus = []
    if html_stripping:
        doc= strip_html_tags(doc)
    # remove accented characters
    if accented_char_removal:
        doc = remove_accented_chars(doc)
    # expand contractions
    if contraction_expansion:
        doc = expand_contractions(doc)
    # lowercase the text
    if text_lower_case:
        doc = doc.lower()
    # remove extra newlines
    doc = re.sub(r'[\r|\n|\r\n]+', ' ',doc)
    # remove special characters and\or digits
    if special_char_removal:
        doc=remove_special_characters(doc,remove_digits=True)
     # remove extra whitespace
    doc = re.sub(' +', ' ', doc)
    # remove stopwords
    if stopword_removal:
        doc = remove_stopwords(doc, is_lower_case=text_lower_case)
    if remove_url:
        doc=url_removal(doc)
    #lemmatize text
    if lemmatize:
        doc = lemmatize_text(doc)
    #stemming text
    if stem:
        doc = stem_text(doc)
    normalized_corpus.append(doc)
    return ' '.join(normalized_corpus)

rawData['tweets_cleaned']= rawData['tweets'].apply(lambda x:normalize_corpus(x))
rawData

C:\Users\ragha\AppData\Local\Temp\ipykernel_16480\172976477.py:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


,tweets,tweets_cleaned
0,@johensley @DarcyShepherd13 @JeffreyGuterman @...,johensley darcyshepherd jeffreyguterman realdo...
1,May i remind you that the vaccine isnt just fo...,may remind vaccin caught covid also prevent
2,"@MJAckermanMDPhD\nHi, Dr. Ackerman. I will lik...",mjackermanmdphd hi dr ackerman like know go di...
3,"Shandro, Hinshaw to give COVID-19 vaccine upda...",shandro hinshaw give covid vaccin updat noon
4,You ever Noticed This? They just announced Yes...,ever notic announc yesterday decemb th great v...
...,...,...
95,Must viewing: Principles of vaccines programs ...,must view principl vaccin program control covi...
96,COVID-19 vaccine's protection against virus ou...,covid vaccin protect viru outweigh potenti all...
97,I've spent some time today looking into whethe...,spent time today look whether covid vaccin do ...
98,COVID-19 Vaccine Likely Beneficial For Breastf...,covid vaccin like benefici breastf babi questi...


In [138]:
import spacy
nltk.download('averaged_perceptron_tagger')
# Download NLTK Punkt sentence tokenizer
nltk.download('punkt')
nlp = spacy.load("en_core_web_sm")
pos_tagged_data=[]
for sentence in rawData['tweets_cleaned']:
    sentence_nlp = nlp(sentence)
    spacy_pos_tagged=[]
    spacy_pos_tagged_sentence = [(word.__str__().strip(' '), word.tag_, word.pos_,sentence) for word in [i for i in sentence_nlp]]
    pos_tagged_data=pos_tagged_data+spacy_pos_tagged_sentence


spacy_pos_tagged_data = pd.DataFrame(pos_tagged_data,columns=['word','pos_tag','tag_type','sentence'])
spacy_pos_tagged_data.to_csv('spacy_pos_tagged_data.csv')
spacy_pos_tagged_data.shape


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


(1624, 4)

## Question 1:

### 1A Frequency of the word covid appears in the dataset:

In [139]:
print("Frequency of the word 'covid' is: {}".format(spacy_pos_tagged_data[spacy_pos_tagged_data['word'].str.contains('covid')]['word'].count()))

Frequency of the word 'covid' is: 118


### 1B Word count of 'covid' across each pos_tag type

In [140]:
spacy_pos_tagged_data['word']=spacy_pos_tagged_data['word'].str.strip(' ')
covid_word_pos_tags = spacy_pos_tagged_data[spacy_pos_tagged_data['word'].str.contains('covid')].groupby(
    ['word','pos_tag']).size().reset_index(name='counts')
covid_word_pos_tags.to_csv('covid_word_pos_tags.csv')
covid_word_pos_tags.head(10)

,word,pos_tag,counts
0,covid,JJ,45
1,covid,NN,27
2,covid,NNP,37
3,covid,VB,2
4,covidindiaseva,NNP,1
5,covidinsa,NN,1
6,covidlong,NNP,1
7,covidnewsbymib,NNP,1
8,covidvaccin,NNP,2
9,longcovid,NNP,1


In [141]:
covid_word_sntns_pos_tags = spacy_pos_tagged_data[spacy_pos_tagged_data['word'].str.contains('covid')].groupby(
    ['word','sentence','pos_tag']).size().reset_index(name='counts')
covid_word_sntns_pos_tags.to_csv('covid_word_sntns_pos_tags.csv')
covid_word_sntns_pos_tags.head(10)

,word,sentence,pos_tag,counts
0,covid,alright guy want make trip pact cuz iatmm summ...,NN,1
1,covid,amoderna expect covid vaccin protect uk corona...,JJ,1
2,covid,ano covid relat death covid case requir medic ...,NN,1
3,covid,ano covid relat death covid case requir medic ...,NNP,1
4,covid,ask answer check updat north carolina covid va...,NNP,1
5,covid,az scheint gut zu wirken afirst dose bntb vacc...,NNP,1
6,covid,b poll plan receiv type covid vaccin covidvaccin,NN,1
7,covid,babi vaccin new studi suggest pfizer covid vac...,NN,1
8,covid,bell palsi convent medicin say caus unknown go...,NNP,1
9,covid,bobwacht peter mark explain chang dose vaccin ...,NN,1


### Question 2:

## 2 Cardinal entity(CD) Count

In [142]:
pos_cardinal_count = spacy_pos_tagged_data[spacy_pos_tagged_data['pos_tag']=='CD'].groupby(["word","pos_tag"]).size().reset_index(name='counts').sort_values('counts',ascending=False)
pos_cardinal_count.to_csv('cardinal_counts.csv')
pos_cardinal_count.head(10)


,word,pos_tag,counts
1,one,CD,5
3,two,CD,4
0,million,CD,1
2,six,CD,1


In [143]:
spacy_pos_tagged_data[spacy_pos_tagged_data['pos_tag']=='CD']

,word,pos_tag,tag_type,sentence
577,million,CD,NUM,vernersview anyon provid number peopl may suff...
636,one,CD,NUM,youatmr look thorough easytounderstand explan ...
847,one,CD,NUM,davek mention earlier tweet learn covid vaccin...
1082,six,CD,NUM,decemb unit state seen six case anaphylaxi amo...
1104,two,CD,NUM,decemb unit state seen six case anaphylaxi amo...
1255,one,CD,NUM,novavax say covid vaccin effect far le one var...
1296,two,CD,NUM,dad demand answer health worker daughter die t...
1332,two,CD,NUM,heathdonsmcgreg mfwitch itatm good skeptic poi...
1395,one,CD,NUM,rrrbyn covid vaccin use one two human fetal ce...
1396,two,CD,NUM,rrrbyn covid vaccin use one two human fetal ce...


### Question 3
<span color:white>Identify the top 5 most commonly appearing entity types in the provided dataset and determine
their respective frequencies using the Spacy Named Entity Recognition (NER).</span>

### NER

In [152]:
ner_tags=[]
for i in rawData['tweets_cleaned']:
    doc = nlp(i)
    if doc.ents:
        ner_tags= ner_tags+[(ent.text,ent.label_,word.tag_) for word,ent in zip(doc,doc.ents)]
    # else:
    #     print(i)

ner_tags_data =pd.DataFrame(ner_tags,columns=['text','label','pos_tag'])
ner_tags_data.head(200)

,text,label,pos_tag
0,johensley darcyshepherd jeffreyguterman,PERSON,NNP
1,second,ORDINAL,NNP
2,efficaci fine,ORG,NNP
3,mrna,GPE,NN
4,dr ackerman,PERSON,NNP
...,...,...,...
114,dr simon,PERSON,JJ
115,potenti allergi,PERSON,JJ
116,today,DATE,VBD
117,summer,DATE,UH


In [156]:
ner_tags_data.groupby(['label','pos_tag']).size().reset_index(name='count').sort_values('count',ascending=False)

,label,pos_tag,count
30,PERSON,NNP,31
24,ORG,NNP,16
29,PERSON,NN,8
28,PERSON,JJ,8
0,CARDINAL,NNP,4
23,ORG,NN,4
18,NORP,NNP,4
12,GPE,NNP,3
31,PERSON,UH,3
3,DATE,JJ,3
